# Predicción de defectos de software

**Curso:** Inteligencia Artificial  
**Docente:** Julian Rene Muñoz Burbano  
**Coleccion:** 1 · **Proyecto:** 4

El modelo estima P(defecto) por modulo con metricas NASA PC1. El semaforo de riesgo lo aplican reglas de negocio (umbrales, modulo critico, cobertura vs LOC).

In [1]:
from pathlib import Path
import sys, json
ROOT=Path.cwd().resolve()
if not (ROOT/'config.json').exists():
    ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
config=json.loads((ROOT/'config.json').read_text(encoding='utf-8'))
config

{'titulo': 'Predicción de defectos de software',
 'coleccion': 1,
 'numero': 4,
 'db': 'ia_c1_p04',
 'tipo_modelo': 'clasificacion_defectos',
 'docente': 'Julian Rene Muñoz Burbano',
 'openml_id': 1068,
 'dataset_nombre': 'pc1',
 'dataset_url': 'https://www.openml.org/d/1068',
 'umbrales_riesgo': {'medio': 0.3, 'alto': 0.6},
 'cobertura_minima': 0.5,
 'loc_sin_cobertura_alto': 200}

## 1. Dataset y ETL

Fuente: NASA PC1 / OpenML 1068. `descargar_dataset.py` obtiene el CSV y `preparar` limpia metricas McCabe/Halstead.

In [2]:
from backend.etl import load_processed, FEATURES, TARGET
from pathlib import Path
import subprocess, sys
if not (Path('data')/'raw'/'pc1.csv').exists():
    subprocess.run([sys.executable, 'descargar_dataset.py'], check=True)
df=load_processed()
df[FEATURES+[TARGET]].describe().T.head(8)

,count,mean,std,min,25%,50%,75%,max
loc,1109.0,23.376105,35.284017,0.0,7.00,13.00,26.00,602.00
v(g),1109.0,5.510730,8.958976,1.0,1.00,3.00,6.00,136.00
ev(g),1109.0,2.766817,5.567704,1.0,1.00,1.00,3.00,123.00
iv(g),1109.0,3.321371,6.401998,1.0,1.00,2.00,3.00,123.00
n,1109.0,117.393417,197.336888,1.0,25.00,58.00,126.00,2785.00
v,1109.0,699.711217,1509.545684,0.0,97.67,275.10,674.04,25942.69
l,1109.0,0.129414,0.146952,0.0,0.05,0.08,0.16,2.00
d,1109.0,15.396267,16.337508,0.0,5.71,11.61,20.50,270.66


## 2. Entrenamiento versionado

Particion estratificada 80/20. LogisticRegression con class_weight=balanced. Artefactos en `models/`.

In [3]:
from backend.ml import train
info=train(df)
info['metrics']

{'roc_auc': 0.8510466988727858,
 'f1': 0.3492063492063492,
 'precision': 0.22916666666666666,
 'recall': 0.7333333333333333,
 'matriz_confusion': {'tn': 170, 'fp': 37, 'fn': 4, 'tp': 11},
 'n_train': 887,
 'n_test': 222,
 'tasa_defectos': 0.06943192064923355,
 'umbral_clase': 0.5}

## 3. Prediccion por modulo (IA + reglas)

In [4]:
from backend.ml import predict
ejemplo={
    'module_id':'auth-service',
    'metricas':{c:float(df[c].median()) for c in FEATURES},
    'cobertura_tests':0.2,
    'es_modulo_critico':True,
}
ejemplo['metricas']['loc']=260
predict(ejemplo)

{'module_id': 'auth-service',
 'probabilidad_defecto': 0.7933,
 'prediccion_clase': 1,
 'explicacion': [{'metrica': 'loc',
   'valor': 260.0,
   'contribucion': 2.7403261974781783,
   'efecto': 'aumenta_riesgo'},
  {'metrica': 'd',
   'valor': 11.61,
   'contribucion': 0.616615837966461,
   'efecto': 'aumenta_riesgo'},
  {'metrica': 'uniq_Opnd',
   'valor': 12.0,
   'contribucion': -0.49542735290044515,
   'efecto': 'reduce_riesgo'}],
 'metricas_imputadas': [],
 'version_modelo': '20261101T151506Z',
 'dataset_hash': '8f4caf03ea356bab',
 'nota': 'IA: probabilidad. Negocio: nivel_riesgo y reglas_aplicadas.',
 'nivel_riesgo': 'alto',
 'nivel_segun_modelo': 'alto',
 'reglas_aplicadas': ['regla: cobertura 20% < 50% y LOC 260 >= 200'],
 'origen_nivel': 'reglas_negocio',
 'umbrales': {'medio': 0.3, 'alto': 0.6}}

## 4. API y dashboard

MongoDB local o `docker compose up`. Terminales: `uvicorn backend.api:app --reload --port 8000` y `streamlit run frontend/app.py`. Documentacion: `/docs`.

Roles: operativo (prediccion), analista (metricas), admin (POST /train).